In [ ]:
# Dev kernel: smoke-test QwenAgent on Kaggle's H100 with the bundled
# Qwen3.6-35B-A3B BF16 weights mounted via dataset_sources.
#
# Pre-reqs:
#   1. Bundler kernel (exp004_qwen_agent/bundle_qwen_kernel) ran COMPLETE.
#   2. Private Kaggle Dataset 'cataluna84/qwen3-6-35b-a3b-bf16' exists.
#   3. competition data attached (gives us /kaggle/input/arc-prize-2026-arc-agi-3/).
#
# This kernel only smoke-tests the loop -- it does NOT write a submission.
# Promotion to a competition kernel happens after the smoke results look healthy.
import os, sys, time, json, subprocess
from pathlib import Path

# --- 1. Locate the mounted Qwen weights -------------------------------------
# When mounted via dataset_sources, the dataset shows up at
# /kaggle/input/<slug>/  (slug = lowercase dataset-id without the user prefix).
QWEN_DIR = Path('/kaggle/input/qwen3-6-35b-a3b-bf16')
if not QWEN_DIR.exists():
    print('[ERROR] Qwen dataset not mounted at', QWEN_DIR)
    print('Available /kaggle/input/ entries:')
    for p in Path('/kaggle/input').iterdir():
        print('  -', p)
    sys.exit(1)

print('Qwen weights at', QWEN_DIR)
print('Total size:', sum(f.stat().st_size for f in QWEN_DIR.rglob('*') if f.is_file())/1e9, 'GB')
for f in sorted(QWEN_DIR.rglob('*'))[:10]:
    if f.is_file():
        print(f'  {f.stat().st_size/1e9:>6.2f} GB  {f.relative_to(QWEN_DIR)}')

# --- 2. Install the arc-agi SDK from the competition wheels ----------------
# The SDK is needed for the real env loop. The wheels are bundled under the
# competition data at /kaggle/input/arc-prize-2026-arc-agi-3/.
WHEELS = Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
if WHEELS.exists():
    print('installing arc-agi + arcengine from', WHEELS)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-index', f'--find-links={WHEELS}',
                    'arc-agi', 'arcengine'], check=False)
else:
    print('[WARN] no wheels at', WHEELS, '- skipping SDK install')

# --- 3. Make our local agents/ package importable --------------------------
# We attach this kernel to a kernel_sources entry that points at our agents
# repo, OR we paste the repo into /kaggle/working at kernel-edit time.
# For v1 we paste the relevant agent files into the kernel via 'kernel_sources'.
AGENTS_PKG = Path('/kaggle/working/agents')
if not AGENTS_PKG.exists():
    # Inline-bootstrap path: attempt to fetch our repo from /kaggle/input/agents/
    src = Path('/kaggle/input/arc-agi-3-agents-pkg/agents')
    if src.exists():
        import shutil
        shutil.copytree(src, AGENTS_PKG)
        print(f'copied agents pkg from {src} -> {AGENTS_PKG}')
    else:
        print('[WARN] agents pkg not found - QwenAgent import will fail')
sys.path.insert(0, '/kaggle/working')

# --- 4. Configure the agent and run on one game ----------------------------
os.environ['QWEN_MODEL_PATH'] = str(QWEN_DIR)
os.environ['QWEN_DTYPE']      = 'bf16'
os.environ['QWEN_DEVICE_MAP'] = 'auto'
os.environ['QWEN_DEBUG_PROMPTS'] = '1'
os.environ['QWEN_MAX_NEW_TOKENS'] = '96'

from agents.qwen_agent import QwenAgent  # noqa: E402
from agents import GameAction, GameState  # noqa: E402

GAME_ID = os.environ.get('SMOKE_GAME', 'ls20')
MAX_ACTIONS = int(os.environ.get('MAX_ACTIONS', 50))

from arc_agi import Arcade  # noqa: E402
arcade = Arcade()
env = arcade.make(GAME_ID)
frame = env.observation_space

agent = QwenAgent(arc_env=env, game_id=GAME_ID)
print(f'starting smoke run on {GAME_ID}, max_actions={MAX_ACTIONS}')

history_log = []
t0 = time.time()
for step in range(MAX_ACTIONS):
    if frame.state in (GameState.WIN, GameState.GAME_OVER):
        print(f'  [step {step}] terminal state {frame.state}, breaking')
        break
    ts = time.time()
    action = agent.choose_action(frame)
    dt = time.time() - ts
    print(f'  [step {step}] {action.name}  state={frame.state}  '
          f'level={frame.levels_completed}  dt={dt:.2f}s')
    history_log.append({'step': step, 'action': action.name,
                        'state': str(frame.state), 'dt': round(dt,3)})
    data = {}
    ad = getattr(action, 'action_data', None)
    if ad is not None:
        try: data = ad.model_dump()
        except Exception: data = getattr(action, '_data', {}) or {}
    next_frame = env.step(action, data=data, reasoning=None)
    if next_frame is None:
        print('  env.step returned None')
        break
    frame = next_frame

elapsed = time.time() - t0
print()
print('=== SMOKE SUMMARY ===')
print(f'  game             : {GAME_ID}')
print(f'  actions taken    : {len(history_log)}')
print(f'  levels_completed : {getattr(frame, "levels_completed", 0)}')
print(f'  win_levels       : {getattr(frame, "win_levels", 0)}')
print(f'  final state      : {frame.state}')
print(f'  wall clock       : {elapsed:.1f}s ({elapsed/max(len(history_log),1):.2f}s/action)')

with open('/kaggle/working/qwen_smoke.json', 'w') as fh:
    json.dump({
        'game': GAME_ID,
        'actions': history_log,
        'levels_completed': getattr(frame, 'levels_completed', 0),
        'final_state': str(frame.state),
        'wall_clock_s': round(elapsed, 2),
    }, fh, indent=2)
print('saved /kaggle/working/qwen_smoke.json')
